# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from src.config import *
from src.utils import *
import datetime

In [48]:
#store start time of notebook
start_time = datetime.datetime.now()
print("Start time: ", start_time)

Start time:  2025-05-25 13:38:51.624441


## 1 Nettoyer la colonne MIN dans les boxscores

In [3]:



# Si tu utilises plusieurs fichiers, adapte avec concat, ici sur un seul batch pour la démo :
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)  # Ou DATA_PLAYERS_DIR selon ton code
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)  # Ou DATA_PLAYERS_DIR selon ton code

print("Boxscores file: ", boxscores_file)
print("Games file: ", games_file)

df_boxscores = pd.read_csv(boxscores_file, low_memory=False, dtype={'GAME_ID': str})
df_games = pd.read_csv(games_file, low_memory=False, dtype={'GAME_ID': str})

print(df_games.columns)

# 3. Merge pour ajouter GAME_DATE à chaque ligne de boxscore
if 'GAME_DATE' not in df_boxscores.columns:
    # Pour éviter les doublons ou les merges en cascade si tu relances le code
    if 'GAME_DATE' in df_games.columns:
        df_boxscores = df_boxscores.merge(
            df_games[['GAME_ID', 'GAME_DATE']],
            on='GAME_ID',
            how='left'
        )
        # Conversion en datetime
        df_boxscores['GAME_DATE'] = pd.to_datetime(df_boxscores['GAME_DATE'])
    else:
        print("No GAME_DATE column found in games_df. Please check your merge operation.")

# Fonction de conversion “mm:ss” -> float minutes
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    if isinstance(val, float) or isinstance(val, int):
        return float(val)
    try:
        parts = str(val).split(':')
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return minutes + seconds/60
        else:
            # Cas où c'est déjà un float/int
            return float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['MIN'].apply(convert_minutes)

print(df_boxscores[['PLAYER_NAME', 'MIN', 'MINUTES_PLAYED']].head(10))
print(df_boxscores['MINUTES_PLAYED'].describe())


Boxscores file:  data/raw_last/batches_merged/merged_boxscores_2025-05-25_10-26-37.csv
Games file:  data/raw_last/games_merged/all__2025-05-25_10-26-37.csv


Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON'],
      dtype='object')
         PLAYER_NAME    MIN  MINUTES_PLAYED
0  Clifford Robinson  42:12       42.200000
1  Clifford Robinson  42:12       42.200000
2       Shawn Marion  28:07       28.116667
3       Shawn Marion  28:07       28.116667
4       Chris Dudley  14:37       14.616667
5       Chris Dudley  14:37       14.616667
6         Mario Elie  10:23       10.383333
7         Mario Elie  10:23       10.383333
8         Jason Kidd  44:46       44.766667
9         Jason Kidd  44:46       44.766667
count    1.708180e+06
mean     1.822104e+01
std      1.395167e+01
min      0.000000e+00
25%      2.983333e+00
50%      1.903333e+01
75%      3.001667e+01
max      6.496667e+01
Name: MINUTES_PLAYED

In [4]:
df_boxscores

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,GAME_DATE,MINUTES_PLAYED
0,0020000011,1610612756,PHX,Phoenix,361,Clifford Robinson,Clifford,G,NaN,42:12,...,5.0,3.0,2.0,1.0,1.0,2.0,26.0,-3.0,2000-10-31,42.200000
1,0020000011,1610612756,PHX,Phoenix,361,Clifford Robinson,Clifford,G,NaN,42:12,...,5.0,3.0,2.0,1.0,1.0,2.0,26.0,-3.0,2000-10-31,42.200000
2,0020000011,1610612756,PHX,Phoenix,1890,Shawn Marion,Shawn,G,NaN,28:07,...,6.0,1.0,1.0,1.0,3.0,3.0,16.0,-12.0,2000-10-31,28.116667
3,0020000011,1610612756,PHX,Phoenix,1890,Shawn Marion,Shawn,G,NaN,28:07,...,6.0,1.0,1.0,1.0,3.0,3.0,16.0,-12.0,2000-10-31,28.116667
4,0020000011,1610612756,PHX,Phoenix,201,Chris Dudley,Chris,F,NaN,14:37,...,2.0,0.0,0.0,0.0,2.0,3.0,2.0,3.0,2000-10-31,14.616667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1708175,0042400313,1610612750,MIN,Minnesota,1631159,Leonard Miller,Leonard,NaN,NaN,7.000000:41,...,3.0,0.0,0.0,0.0,1.0,0.0,11.0,0.0,2025-05-24,0.000000
1708176,0042400313,1610612750,MIN,Minnesota,1630568,Luka Garza,Luka,NaN,NaN,5.000000:06,...,0.0,0.0,0.0,0.0,0.0,0.0,7.0,-2.0,2025-05-24,0.000000
1708177,0042400313,1610612750,MIN,Minnesota,1630568,Luka Garza,Luka,NaN,NaN,5.000000:06,...,0.0,0.0,0.0,0.0,0.0,0.0,7.0,-2.0,2025-05-24,0.000000
1708178,0042400313,1610612750,MIN,Minnesota,204060,Joe Ingles,Joe,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-24,0.000000


## Type correction

In [5]:
cols_to_float = [
    'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 
    'FTM', 'FTA', 'OREB', 'DREB', 'PLUS_MINUS'
]
for col in cols_to_float:
    if col in df_boxscores.columns:
        df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)


 ### Optionnel : repérage/ajout d’un flag “Starter”

In [6]:
df_boxscores['IS_STARTER'] = df_boxscores['START_POSITION'].notna() & (df_boxscores['START_POSITION'] != '')

## Étape 2 — Agrégation de stats par équipe et par match

In [7]:
agg_cols = [
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED'
    ]


team_match_stats = df_boxscores.groupby(['GAME_ID', 'TEAM_ID', 'GAME_DATE'])[agg_cols].sum().reset_index()

# Optionnel : rajoute l’équipe adverse dans chaque ligne pour faciliter le merge futur
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1
)



In [8]:
team_match_stats


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000,1610612755
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000,1610612752
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000,1610612751
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000,1610612739
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,74.0,40.0,20.0,18.0,30.0,48.0,194.0,110.0,480.000000,1610612764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0052300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,98.0,56.0,20.0,10.0,16.0,34.0,236.0,240.0,480.033333,1610612744
64142,0052300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000,1610612748
64143,0052300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000,1610612741
64144,0052300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,90.0,58.0,18.0,16.0,30.0,38.0,210.0,70.0,480.066667,1610612758


##  Renommer les colonnes pour merge

In [9]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]

df_team = team_match_stats.copy()
df_opp = team_match_stats.copy()
df_opp = df_opp.rename(
    columns={col: f"OPP_{col}" for col in team_cols}
).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'}
)

df_team
df_opp

,GAME_ID,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,...,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,TEAM_ID
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000,1610612755
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000,1610612752
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000,1610612751
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000,1610612739
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,74.0,40.0,20.0,18.0,30.0,48.0,194.0,110.0,480.000000,1610612764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0052300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,98.0,56.0,20.0,10.0,16.0,34.0,236.0,240.0,480.033333,1610612744
64142,0052300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000,1610612748
64143,0052300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000,1610612741
64144,0052300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,90.0,58.0,18.0,16.0,30.0,38.0,210.0,70.0,480.066667,1610612758


## Merge

In [10]:
match_dataset = pd.merge(
    df_team,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [11]:
match_dataset 
match_dataset.shape
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,70.0,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,82.0,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,70.0,88.0,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0052300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,68.0,84.0,38.0,10.0,6.0,32.0,34.0,188.0,-240.0,480.000000
64142,0052300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000
64143,0052300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,62.0,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000
64144,0052300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,52.0,80.0,42.0,18.0,8.0,30.0,34.0,196.0,-70.0,480.000000


In [12]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED',
       'OPP_TEAM_ID', 'OPP_GAME_DATE', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
       'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTM', 'OPP_FTA',
       'OPP_FT_PCT', 'OPP_OREB', 'OPP_DREB', 'OPP_REB', 'OPP_AST', 'OPP_STL',
       'OPP_BLK', 'OPP_TO', 'OPP_PF', 'OPP_PTS', 'OPP_PLUS_MINUS',
       'OPP_MINUTES_PLAYED'],
      dtype='object')

In [13]:
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,70.0,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,82.0,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,70.0,88.0,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,112.0,164.0,80.0,8.0,28.0,56.0,80.0,412.0,-300.0,0.0
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,100.0,128.0,104.0,24.0,12.0,40.0,92.0,456.0,100.0,0.0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,112.0,152.0,72.0,24.0,16.0,48.0,76.0,436.0,-100.0,0.0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,104.0,124.0,76.0,20.0,12.0,56.0,80.0,404.0,-840.0,0.0


## Add Win/Loose flag

In [14]:
# Garde l’info du score
match_dataset['IS_WIN'] = (match_dataset['PTS'] > match_dataset['OPP_PTS']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['PTS'] - match_dataset['OPP_PTS']

match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,IS_WIN,POINT_DIFF
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0,0,-58.0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0,1,58.0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.0,1,8.0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.0,0,-8.0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.0,1,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,80.0,8.0,28.0,56.0,80.0,412.0,-300.0,0.0,1,60.0
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,104.0,24.0,12.0,40.0,92.0,456.0,100.0,0.0,0,-20.0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,72.0,24.0,16.0,48.0,76.0,436.0,-100.0,0.0,1,20.0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,76.0,20.0,12.0,56.0,80.0,404.0,-840.0,0.0,1,168.0


## Add IS_HOME flag

In [15]:
mask_home = df_games['MATCHUP'].str.contains('vs\.')
mask_away = df_games['MATCHUP'].str.contains('@')
mask_none = ~(mask_home | mask_away)  # Ni l’un ni l’autre

# Affiche le nombre de cas problématiques (doit être 0 normalement)
print(f"Lignes indéterminées : {mask_none.sum()}")

# Si tu veux voir lesquelles
if mask_none.sum() > 0:
    print(df_games.loc[mask_none, ['GAME_ID', 'TEAM_ABBREVIATION', 'MATCHUP']])


Lignes indéterminées : 0


In [16]:
if 'MATCHUP' in match_dataset.columns:
    match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs\.').astype(int)
else:
    # Faut merge avec df_games pour récupérer MATCHUP
    match_dataset = match_dataset.merge(
        df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP']],
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )
    match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs\.').astype(int)
    
#remove MATCHUP column because we don't need it anymore
match_dataset = match_dataset.drop(columns=['MATCHUP'])


In [17]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,IS_WIN,POINT_DIFF,IS_HOME
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,20.0,10.0,26.0,48.0,202.0,290.0,480.0,0,-58.0,1
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,12.0,8.0,44.0,60.0,144.0,-290.0,480.0,1,58.0,0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,18.0,16.0,24.0,62.0,164.0,-40.0,480.0,1,8.0,0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,10.0,16.0,38.0,54.0,172.0,40.0,480.0,0,-8.0,1
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,12.0,2.0,52.0,56.0,172.0,-110.0,480.0,1,22.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,8.0,28.0,56.0,80.0,412.0,-300.0,0.0,1,60.0,1
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,24.0,12.0,40.0,92.0,456.0,100.0,0.0,0,-20.0,1
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,24.0,16.0,48.0,76.0,436.0,-100.0,0.0,1,20.0,0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,20.0,12.0,56.0,80.0,404.0,-840.0,0.0,1,168.0,1


## Rolling winrate on N matchs and global

In [18]:
N_LIST = [5, 10, 25, 50, 100, 200]

for n in N_LIST:
    # Rolling winrate à domicile
    match_dataset[f'ROLL_HOME_WINRATE_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['IS_WIN'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )

    # Rolling winrate à l'extérieur
    match_dataset[f'ROLL_AWAY_WINRATE_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['IS_WIN'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [19]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,0.800000,0.600000,0.866667,0.700000,0.821429,0.772727,0.823529,0.755102,0.810000,0.650000
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,0.400000,0.800000,0.571429,0.636364,0.629630,0.608696,0.647059,0.591837,0.653465,0.555556
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,0.750000,0.833333,0.800000,0.800000,0.807692,0.583333,0.714286,0.509804,0.663265,0.490196
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,0.800000,0.600000,0.750000,0.615385,0.680000,0.600000,0.571429,0.568627,0.642857,0.598039


## Streaks à domicile / extérieur

In [20]:
def calc_streak_home_away(results, is_home):
    streak_home = []
    streak_away = []
    cur_home = cur_away = 0
    for r, h in zip(results, is_home):
        if h == 1:
            if r == 1:
                cur_home += 1
            else:
                cur_home = 0
            streak_home.append(cur_home)
            streak_away.append(cur_away)
        else:
            if r == 1:
                cur_away += 1
            else:
                cur_away = 0
            streak_away.append(cur_away)
            streak_home.append(cur_home)
    return streak_home, streak_away

grouped = match_dataset.groupby('TEAM_ID')

# Shift IS_WIN et IS_HOME AVANT le calcul !
match_dataset['IS_WIN_SHIFTED'] = grouped['IS_WIN'].shift(1).fillna(0).astype(int)
match_dataset['IS_HOME_SHIFTED'] = grouped['IS_HOME'].shift(1).fillna(0).astype(int)

home_streaks = []
away_streaks = []
for _, df in grouped:
    home, away = calc_streak_home_away(df['IS_WIN_SHIFTED'].values, df['IS_HOME_SHIFTED'].values)
    home_streaks.extend(home)
    away_streaks.extend(away)

match_dataset['HOME_WIN_STREAK'] = home_streaks
match_dataset['AWAY_WIN_STREAK'] = away_streaks

# Clean up
match_dataset = match_dataset.drop(columns=['IS_WIN_SHIFTED', 'IS_HOME_SHIFTED'])



In [21]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,HOME_WIN_STREAK,AWAY_WIN_STREAK
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,0.866667,0.700000,0.821429,0.772727,0.823529,0.755102,0.810000,0.650000,1,0
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,0.571429,0.636364,0.629630,0.608696,0.647059,0.591837,0.653465,0.555556,1,0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,0.800000,0.800000,0.807692,0.583333,0.714286,0.509804,0.663265,0.490196,0,0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,0.750000,0.615385,0.680000,0.600000,0.571429,0.568627,0.642857,0.598039,0,0


##  Moyennes offensives/défensives home/away

In [22]:
for n in N_LIST:
    # Points marqués à domicile
    match_dataset[f'ROLL_HOME_PTS_FOR_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['PTS'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    # Points encaissés à domicile
    match_dataset[f'ROLL_HOME_PTS_AGAINST_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['OPP_PTS'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    # Idem pour l'extérieur
    match_dataset[f'ROLL_AWAY_PTS_FOR_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['PTS'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    match_dataset[f'ROLL_AWAY_PTS_AGAINST_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['OPP_PTS'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [23]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_PTS_AGAINST_200
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,235.363636,224.181818,248.509804,216.117647,230.163265,213.714286,266.140000,236.520000,245.240000,235.340000
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,218.000000,222.347826,240.039216,227.294118,224.530612,224.612245,246.435644,233.009901,239.454545,237.414141
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,250.750000,246.666667,231.306122,219.632653,238.235294,239.176471,266.142857,253.530612,248.921569,252.411765
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,238.720000,238.640000,228.938776,216.489796,227.372549,225.725490,244.755102,228.979592,243.098039,237.725490


## Rolling Features on N matchs

In [24]:
for stat in ['PTS', 'REB', 'AST', 'FGM', 'FGA', 'FG_PCT', 'PLUS_MINUS']:
    for n in [3, 5, 10, 25, 50, 100,200]:
        match_dataset[f'ROLL_{stat}_{n}'] = (
            match_dataset
            .sort_values(['TEAM_ID', 'GAME_DATE'])
            .groupby('TEAM_ID')[stat]
            .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
        )


match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,10.02396,9.72074,10.73602,240.000000,168.0,124.0,148.0,132.4,122.9,98.62
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,8.46376,8.43140,8.93332,23.333333,-14.0,-2.0,17.2,15.2,32.3,38.95
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,9.82364,9.87584,10.87769,116.666667,28.0,42.0,72.0,49.2,26.2,22.00
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,9.07856,8.80796,9.97337,-236.666667,-118.0,-24.0,43.6,50.6,34.7,52.35


## Streaks et Win Ratio

In [25]:
for n in [3, 5, 10, 25, 50, 100, 200]:
    match_dataset[f'ROLL_WIN_RATIO_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')['IS_WIN']
        .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    )


In [26]:
def calc_win_streak(results):
    streak = []
    cur = 0
    for r in results:
        if r == 1:
            cur += 1
        else:
            cur = 0
        streak.append(cur)
    return streak

# >>> Correction : shift IS_WIN avant de calculer le streak
match_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['IS_WIN_SHIFTED'] = (
    match_dataset
    .groupby('TEAM_ID')['IS_WIN']
    .shift(1)
    .fillna(0)
    .astype(int)
)

match_dataset['WIN_STREAK'] = (
    match_dataset
    .groupby('TEAM_ID')['IS_WIN_SHIFTED']
    .transform(calc_win_streak)
)



match_dataset = match_dataset.sort_values(['GAME_DATE','GAME_ID']).reset_index(drop=True)

In [27]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_PLUS_MINUS_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,IS_WIN_SHIFTED,WIN_STREAK
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,98.62,0.666667,0.8,0.7,0.80,0.80,0.79,0.730,1,2
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,38.95,0.333333,0.4,0.6,0.60,0.62,0.62,0.605,0,0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,22.00,1.000000,0.8,0.8,0.80,0.70,0.61,0.575,1,3
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,52.35,0.333333,0.6,0.7,0.68,0.64,0.57,0.620,0,0


## Rest Days & Advantage

In [28]:
match_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7) # Si NaN = 7 jours par défaut
)

match_dataset['OPP_DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('OPP_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7)
)
match_dataset['REST_ADVANTAGE'] = match_dataset['DAYS_SINCE_LAST_GAME'] - match_dataset['OPP_DAYS_SINCE_LAST_GAME']
ch_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7) # Si NaN = 7 jours par défaut
)

match_dataset['OPP_DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('OPP_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7)
)
match_dataset['REST_ADVANTAGE'] = match_dataset['DAYS_SINCE_LAST_GAME'] - match_dataset['OPP_DAYS_SINCE_LAST_GAME']


In [29]:
#group again match_dataset by game_id and game_date

match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,IS_WIN_SHIFTED,WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8880.0,8887.0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8970.0,8977.0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8904.0,8911.0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8868.0,8875.0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8908.0,8915.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,0.7,0.80,0.80,0.79,0.730,1,2,2.0,2.0,0.0
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,0.6,0.60,0.62,0.62,0.605,0,0,2.0,2.0,0.0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,0.8,0.80,0.70,0.61,0.575,1,3,2.0,2.0,0.0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,0.7,0.68,0.64,0.57,0.620,0,0,2.0,2.0,0.0


## Rest advantage home/away

In [30]:
for n in N_LIST:
    match_dataset[f'ROLL_HOME_REST_ADV_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['REST_ADVANTAGE'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    match_dataset[f'ROLL_AWAY_REST_ADV_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['REST_ADVANTAGE'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [31]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,-26.000000,0.000000,-46.133333,-26.000000,-57.071429,-56.909091,-112.745098,-94.959184,-120.760000,-98.280000
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,-18.800000,-4.600000,-44.142857,-33.636364,-89.185185,-74.608696,-95.784314,-110.979592,-92.742574,-117.636364
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,0.000000,-18.333333,-42.800000,-65.500000,-71.307692,-97.041667,-73.775510,-128.294118,-95.826531,-116.245098
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,-21.000000,-15.800000,-60.750000,-51.384615,-94.040000,-59.320000,-109.510204,-95.333333,-105.806122,-103.872549


## H2H - Head To Head WinRate (Last N games)

In [32]:
import pandas as pd
from collections import defaultdict, deque

# On suppose que match_dataset contient déjà une ligne par équipe/match (après le merge et le tri)
# Et que tu as bien trié par date/match
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)

# On prépare les features H2H rolling pour chaque équipe face à chaque adversaire
N_LIST = [5, 10, 25, 50, 100, 200]  # tu peux réduire à ce que tu veux
for N in N_LIST:
    # Dictionnaire: (team, opponent) -> deque des derniers résultats (1 = win, 0 = lose)
    last_n_results = defaultdict(lambda: deque(maxlen=N))
    h2h_diff_list = []
    h2h_winrate_list = []
    h2h_count_list = []

    # Pour chaque ligne (match), on met à jour le rolling H2H
    for idx, row in match_dataset.iterrows():
        team = row['TEAM_ID']
        opp = row['OPP_TEAM_ID']
        team_pts = row['PTS']
        opp_pts = row['OPP_PTS']

        # Calcul du résultat précédent
        history = last_n_results[(team, opp)]
        n_prev = len(history)
        winrate = sum(history) / n_prev if n_prev > 0 else 0.5
        diff = sum(history) - (n_prev - sum(history)) if n_prev > 0 else 0  # nb_victoires - nb_défaites

        h2h_diff_list.append(diff)
        h2h_winrate_list.append(winrate)
        h2h_count_list.append(n_prev)

        # Maj du résultat courant (après l'utilisation pour ne pas polluer la ligne actuelle)
        last_n_results[(team, opp)].append(1 if team_pts > opp_pts else 0)

    match_dataset[f'H2H_LAST_{N}_DIFF'] = h2h_diff_list
    match_dataset[f'H2H_LAST_{N}_WINRATE'] = h2h_winrate_list
    match_dataset[f'H2H_LAST_{N}_COUNT'] = h2h_count_list



In [33]:
match_dataset.tail(10)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT
64136,0042400311,1610612750,2025-05-20,116.0,332.0,15.352,60.0,204.0,8.852,60.0,...,25,-4,0.46,50,-16,0.418367,98,-16,0.418367,98
64137,0042400311,1610612760,2025-05-20,164.0,328.0,26.368,44.0,84.0,22.664,84.0,...,25,4,0.54,50,16,0.581633,98,16,0.581633,98
64138,0042400301,1610612752,2025-05-21,192.0,376.0,16.876,44.0,136.0,7.932,112.0,...,25,-10,0.40,50,-20,0.400000,100,-20,0.401961,102
64139,0042400301,1610612754,2025-05-21,204.0,400.0,17.672,60.0,148.0,8.556,84.0,...,25,10,0.60,50,20,0.600000,100,20,0.598039,102
64140,0042400312,1610612750,2025-05-22,144.0,348.0,12.476,44.0,156.0,7.776,80.0,...,25,-4,0.46,50,-17,0.414141,99,-17,0.414141,99
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,25,4,0.54,50,17,0.585859,99,17,0.585859,99
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,25,-12,0.38,50,-22,0.390000,100,-21,0.398058,103
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,25,12,0.62,50,22,0.610000,100,21,0.601942,103
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,25,-4,0.46,50,-18,0.410000,100,-18,0.410000,100
64145,0042400313,1610612760,2025-05-24,140.0,344.0,23.300,56.0,176.0,17.864,68.0,...,25,4,0.54,50,18,0.590000,100,18,0.590000,100


## H2H points diff (N last games)

In [34]:
for N in N_LIST:
    last_n_pts_for = defaultdict(lambda: deque(maxlen=N))
    last_n_pts_against = defaultdict(lambda: deque(maxlen=N))
    pts_for_list = []
    pts_against_list = []
    margin_list = []

    for idx, row in match_dataset.iterrows():
        team = row['TEAM_ID']
        opp = row['OPP_TEAM_ID']
        team_pts = row['PTS']
        opp_pts = row['OPP_PTS']

        history_for = last_n_pts_for[(team, opp)]
        history_against = last_n_pts_against[(team, opp)]
        n_prev = len(history_for)

        avg_for = sum(history_for)/n_prev if n_prev > 0 else 0
        avg_against = sum(history_against)/n_prev if n_prev > 0 else 0
        avg_margin = (sum(history_for) - sum(history_against))/n_prev if n_prev > 0 else 0

        pts_for_list.append(avg_for)
        pts_against_list.append(avg_against)
        margin_list.append(avg_margin)

        # Update with the current match AFTER calculation
        last_n_pts_for[(team, opp)].append(team_pts)
        last_n_pts_against[(team, opp)].append(opp_pts)

    match_dataset[f'H2H_LAST_{N}_PTS_FOR'] = pts_for_list
    match_dataset[f'H2H_LAST_{N}_PTS_AGAINST'] = pts_against_list
    match_dataset[f'H2H_LAST_{N}_MARGIN'] = margin_list


In [35]:
match_dataset



#print all columns
 #print(match_dataset.columns)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_25_MARGIN,H2H_LAST_50_PTS_FOR,H2H_LAST_50_PTS_AGAINST,H2H_LAST_50_MARGIN,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,-2.32,229.52,226.88,2.64,216.323232,214.141414,2.181818,216.323232,214.141414,2.181818
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,-3.20,216.80,225.40,-8.60,205.900000,213.000000,-7.100000,204.757282,211.553398,-6.796117
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,3.20,225.40,216.80,8.60,213.000000,205.900000,7.100000,211.553398,204.757282,6.796117
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,0.40,230.68,233.88,-3.20,216.120000,218.880000,-2.760000,216.120000,218.880000,-2.760000


In [36]:
print(df_games[['GAME_ID', 'TEAM_ID']].duplicated().sum())  # doit donner 0

0


## H2H Rolling par Saison

In [37]:
# Assure-toi d’avoir une colonne SEASON dans match_dataset, sinon merge avec df_games
if 'SEASON' not in match_dataset.columns:
    # Tu peux ajouter la colonne depuis df_games
    match_dataset = match_dataset.merge(
        df_games[['GAME_ID', 'TEAM_ID', 'SEASON']].drop_duplicates(),
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )



# Création d’un compteur par saison
h2h_season_wins = defaultdict(int)
h2h_season_matches = defaultdict(int)
season_win_counts = []
season_match_counts = []

for idx, row in match_dataset.iterrows():
    season = row['SEASON']
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']    
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']

    win_count = h2h_season_wins[(team, opp, season)]
    match_count = h2h_season_matches[(team, opp, season)]

    season_win_counts.append(win_count)
    season_match_counts.append(match_count)

    # Update counters
    h2h_season_matches[(team, opp, season)] += 1
    if team_pts > opp_pts:
        h2h_season_wins[(team, opp, season)] += 1

match_dataset["H2H_SEASON_WINS"] = season_win_counts
match_dataset["H2H_SEASON_MATCHES"] = season_match_counts
match_dataset["H2H_SEASON_WINRATE"] = [
    w / m if m > 0 else 0 for w, m in zip(season_win_counts, season_match_counts)
]


In [38]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,216.323232,214.141414,2.181818,216.323232,214.141414,2.181818,2024-25,3,5,0.600000
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,205.900000,213.000000,-7.100000,204.757282,211.553398,-6.796117,2024-25,2,4,0.500000
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,213.000000,205.900000,7.100000,211.553398,204.757282,6.796117,2024-25,2,4,0.500000
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,216.120000,218.880000,-2.760000,216.120000,218.880000,-2.760000,2024-25,2,6,0.333333


## H2H Streaks

In [39]:
h2h_streaks = defaultdict(int)
current_streak = defaultdict(int)
last_result = defaultdict(lambda: None)
streak_list = []

for idx, row in match_dataset.iterrows():
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']

    key = (team, opp)

    # Récupère le streak précédent
    streak = current_streak[key]
    last = last_result[key]

    streak_list.append(streak)

    # Update: Si victoire, on incrémente, sinon on reset à 0
    if team_pts > opp_pts:
        if last == "W":
            current_streak[key] += 1
        else:
            current_streak[key] = 1
        last_result[key] = "W"
    else:
        current_streak[key] = 0
        last_result[key] = "L"

match_dataset['H2H_WIN_STREAK'] = streak_list
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,214.141414,2.181818,216.323232,214.141414,2.181818,2024-25,3,5,0.600000,1
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,213.000000,-7.100000,204.757282,211.553398,-6.796117,2024-25,2,4,0.500000,0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,205.900000,7.100000,211.553398,204.757282,6.796117,2024-25,2,4,0.500000,1
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,218.880000,-2.760000,216.120000,218.880000,-2.760000,2024-25,2,6,0.333333,0


## Calcul du ELO global

In [40]:
# --- ELO GLOBAL CALCULATION ---

from collections import defaultdict

# 1. Trie les matchs par date pour garder la cohérence chronologique
match_dataset = match_dataset.sort_values(by=['GAME_DATE', 'GAME_ID', 'TEAM_ID']).reset_index(drop=True)

# 2. Paramètres du ELO
elo_start = 1500
k_factor = 24

# 3. Dictionnaire pour stocker l'ELO de chaque équipe
elo_history = defaultdict(lambda: elo_start)
elo_hist_list = []

# 4. Calcul de l'ELO avant chaque match (pour chaque équipe)
for idx, row in match_dataset.iterrows():
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    game_id = row['GAME_ID']
    
    team_elo_pre = elo_history[team]
    opp_elo_pre = elo_history[opp]
    
    # Stocke les valeurs PRE-match (avant update)
    elo_hist_list.append({
        'GAME_ID': game_id,
        'TEAM_ID': team,
        'ELO_PRE': team_elo_pre
    })
    
    # Calcul du résultat du match
    # On considère 1 pour victoire, 0 pour défaite (pas de draw NBA)
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']
    if team_pts > opp_pts:
        outcome = 1
    else:
        outcome = 0
    
    # Probabilité attendue de victoire
    expected = 1 / (1 + 10 ** ((opp_elo_pre - team_elo_pre) / 400))
    
    # MAJ de l'ELO de l'équipe après le match
    new_elo = team_elo_pre + k_factor * (outcome - expected)
    elo_history[team] = new_elo

# 5. Crée un DataFrame des ELO_PRE par équipe et par match
elo_df = pd.DataFrame(elo_hist_list)

# 6. Ajoute ELO_PRE pour chaque équipe
match_dataset = match_dataset.merge(
    elo_df,
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

# 7. Ajoute ELO_PRE de l'adversaire (OPP_ELO_PRE) en croisant sur OPP_TEAM_ID
match_dataset = match_dataset.merge(
    elo_df.rename(columns={'TEAM_ID': 'OPP_TEAM_ID', 'ELO_PRE': 'OPP_ELO_PRE'}),
    on=['GAME_ID', 'OPP_TEAM_ID'],
    how='left'
)

# 8. (Optionnel) Vérification rapide
print(match_dataset[['GAME_ID', 'TEAM_ID', 'ELO_PRE', 'OPP_TEAM_ID', 'OPP_ELO_PRE']].tail(10))


          GAME_ID     TEAM_ID      ELO_PRE  OPP_TEAM_ID  OPP_ELO_PRE
64136  0042400311  1610612750  1663.571671   1610612760  1767.780936
64137  0042400311  1610612760  1767.780936   1610612750  1663.571671
64138  0042400301  1610612752  1637.560647   1610612754  1666.067182
64139  0042400301  1610612754  1666.067182   1610612752  1637.560647
64140  0042400312  1610612750  1655.066745   1610612760  1776.018986
64141  0042400312  1610612760  1776.018986   1610612750  1655.066745
64142  0042400302  1610612752  1626.543025   1610612754  1676.707924
64143  0042400302  1610612754  1676.707924   1610612752  1626.543025
64144  0042400313  1610612750  1647.083323   1610612760  1783.759474
64145  0042400313  1610612760  1783.759474   1610612750  1647.083323


In [41]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,216.323232,214.141414,2.181818,2024-25,3,5,0.600000,1,1776.018986,1655.066745
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,204.757282,211.553398,-6.796117,2024-25,2,4,0.500000,0,1626.543025,1676.707924
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,211.553398,204.757282,6.796117,2024-25,2,4,0.500000,1,1676.707924,1626.543025
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,216.120000,218.880000,-2.760000,2024-25,2,6,0.333333,0,1647.083323,1783.759474


## Calcul du ELO par saison

In [42]:
from collections import defaultdict

# On part d'un dataset déjà trié par GAME_DATE, SEASON_ID, GAME_ID, TEAM_ID
match_dataset = match_dataset.sort_values(by=['SEASON', 'GAME_DATE', 'GAME_ID', 'TEAM_ID']).reset_index(drop=True)

elo_start = 1500
k_factor = 24

elo_history_season = defaultdict(lambda: elo_start)
elo_hist_list_season = []

for idx, row in match_dataset.iterrows():
    season = row['SEASON']
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    game_id = row['GAME_ID']
    
    # Clés différentes pour chaque saison
    key_team = (season, team)
    key_opp = (season, opp)
    
    team_elo_pre = elo_history_season[key_team]
    opp_elo_pre = elo_history_season[key_opp]
    
    # Stocke avant update
    elo_hist_list_season.append({
        'GAME_ID': game_id,
        'TEAM_ID': team,
        'SEASON': season,
        'ELO_PRE_SEASON': team_elo_pre
    })
    
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']
    outcome = 1 if team_pts > opp_pts else 0
    expected = 1 / (1 + 10 ** ((opp_elo_pre - team_elo_pre) / 400))
    new_elo = team_elo_pre + k_factor * (outcome - expected)
    elo_history_season[key_team] = new_elo

elo_season_df = pd.DataFrame(elo_hist_list_season)

# Ajoute ELO_PRE_SEASON pour chaque équipe
match_dataset = match_dataset.merge(
    elo_season_df,
    on=['GAME_ID', 'TEAM_ID', 'SEASON'],
    how='left'
)

# Ajoute ELO_PRE_SEASON de l'adversaire (OPP_ELO_PRE_SEASON)
match_dataset = match_dataset.merge(
    elo_season_df.rename(columns={'TEAM_ID': 'OPP_TEAM_ID', 'ELO_PRE_SEASON': 'OPP_ELO_PRE_SEASON'}),
    on=['GAME_ID', 'OPP_TEAM_ID', 'SEASON'],
    how='left'
)

# Vérification rapide
print(match_dataset[['GAME_ID', 'SEASON', 'TEAM_ID', 'ELO_PRE_SEASON', 'OPP_TEAM_ID', 'OPP_ELO_PRE_SEASON']].tail(10))


          GAME_ID   SEASON     TEAM_ID  ELO_PRE_SEASON  OPP_TEAM_ID  \
64136  0042400311  2024-25  1610612750     1649.011055   1610612760   
64137  0042400311  2024-25  1610612760     1746.913420   1610612750   
64138  0042400301  2024-25  1610612752     1629.979699   1610612754   
64139  0042400301  2024-25  1610612754     1662.084166   1610612752   
64140  0042400312  2024-25  1610612750     1640.305738   1610612760   
64141  0042400312  2024-25  1610612760     1755.342687   1610612750   
64142  0042400302  2024-25  1610612752     1619.085403   1610612754   
64143  0042400302  2024-25  1610612754     1672.606577   1610612752   
64144  0042400313  2024-25  1610612750     1632.139876   1610612760   
64145  0042400313  2024-25  1610612760     1763.257237   1610612750   

       OPP_ELO_PRE_SEASON  
64136         1746.913420  
64137         1649.011055  
64138         1662.084166  
64139         1629.979699  
64140         1755.342687  
64141         1640.305738  
64142         1672.606

In [49]:
pd.options.display.max_columns = None

match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,IS_WIN,POINT_DIFF,IS_HOME,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,HOME_WIN_STREAK,AWAY_WIN_STREAK,ROLL_HOME_PTS_FOR_5,ROLL_HOME_PTS_AGAINST_5,ROLL_AWAY_PTS_FOR_5,ROLL_AWAY_PTS_AGAINST_5,ROLL_HOME_PTS_FOR_10,ROLL_HOME_PTS_AGAINST_10,ROLL_AWAY_PTS_FOR_10,ROLL_AWAY_PTS_AGAINST_10,ROLL_HOME_PTS_FOR_25,ROLL_HOME_PTS_AGAINST_25,ROLL_AWAY_PTS_FOR_25,ROLL_AWAY_PTS_AGAINST_25,ROLL_HOME_PTS_FOR_50,ROLL_HOME_PTS_AGAINST_50,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_PTS_AGAINST_200,ROLL_PTS_3,ROLL_PTS_5,ROLL_PTS_10,ROLL_PTS_25,ROLL_PTS_50,ROLL_PTS_100,ROLL_PTS_200,ROLL_REB_3,ROLL_REB_5,ROLL_REB_10,ROLL_REB_25,ROLL_REB_50,ROLL_REB_100,ROLL_REB_200,ROLL_AST_3,ROLL_AST_5,ROLL_AST_10,ROLL_AST_25,ROLL_AST_50,ROLL_AST_100,ROLL_AST_200,ROLL_FGM_3,ROLL_FGM_5,ROLL_FGM_10,ROLL_FGM_25,ROLL_FGM_50,ROLL_FGM_100,ROLL_FGM_200,ROLL_FGA_3,ROLL_FGA_5,ROLL_FGA_10,ROLL_FGA_25,ROLL_FGA_50,ROLL_FGA_100,ROLL_FGA_200,ROLL_FG_PCT_3,ROLL_FG_PCT_5,ROLL_FG_PCT_10,ROLL_FG_PCT_25,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,IS_WIN_SHIFTED,WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_LAST_5_PTS_FOR,H2H_LAST_5_PTS_AGAINST,H2H_LAST_5_MARGIN,H2H_LAST_10_PTS_FOR,H2H_LAST_10_PTS_AGAINST,H2H_LAST_10_MARGIN,H2H_LAST_25_PTS_FOR,H2H_LAST_25_PTS_AGAINST,H2H_LAST_25_MARGIN,H2H_LAST_50_PTS_FOR,H2H_LAST_50_PTS_AGAINST,H2H_LAST_50_MARGIN,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,48.0,8.214,28.0,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,60.0,8.358,16.0,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0,0,-58.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8880.0,8887.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.5,0,0,0.5,0,0,0.50,0,0,0.50,0,0,0.500000,0,0,0.500000,0,0.0,0.0,0.0,0.0

## Save final dataset in CSV 

In [44]:

# Chemin de sauvegarde (modifie selon ta structure de dossiers)
final_date = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
final_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')

match_dataset.to_csv(final_path, index=False)
print(f'✅ Dataset sauvegardé à {final_path}')


✅ Dataset sauvegardé à data/final_dataset/nba_features_final_2025-05-25_11-52-00.csv


## Clean final dataset for training and predictions

In [45]:
# Liste des colonnes à dropper (toutes les stats brutes et colonnes de match, identifiants inutiles, etc.)
drop_cols = [
    # Identifiants et logs
    "GAME_ID", "OPP_GAME_DATE", #"OPP_TEAM_ID", "GAME_DATE",

    # Stats brutes de match (pour les deux équipes)
    "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "PTS", "PLUS_MINUS", "MINUTES_PLAYED",

    "OPP_FGM", "OPP_FGA", "OPP_FG_PCT", "OPP_FG3M", "OPP_FG3A", "OPP_FG3_PCT", "OPP_FTM", "OPP_FTA", "OPP_FT_PCT",
    "OPP_OREB", "OPP_DREB", "OPP_REB", "OPP_AST", "OPP_STL", "OPP_BLK", "OPP_TO", "OPP_PF", "OPP_PTS",
    "OPP_PLUS_MINUS", "OPP_MINUTES_PLAYED",
    "POINT_DIFF"
]

# Droppage effectif
final_dataset = match_dataset.drop(columns=[col for col in drop_cols if col in match_dataset.columns])

# Sauvegarde finale
final_cleaned_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

final_dataset.to_csv(final_cleaned_path, index=False)
print(f'✅ Dataset sauvegardé à {final_cleaned_path}')


✅ Dataset sauvegardé à data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-25_11-52-00.csv


In [46]:
for col in final_dataset.columns:
    print(f"{col}: {final_dataset[col].isnull().sum()}")  # Affiche le nombre de NaN par colonne

TEAM_ID: 0
GAME_DATE: 0
OPP_TEAM_ID: 0
IS_WIN: 0
IS_HOME: 0
ROLL_HOME_WINRATE_5: 1701
ROLL_AWAY_WINRATE_5: 1393
ROLL_HOME_WINRATE_10: 54
ROLL_AWAY_WINRATE_10: 53
ROLL_HOME_WINRATE_25: 52
ROLL_AWAY_WINRATE_25: 53
ROLL_HOME_WINRATE_50: 52
ROLL_AWAY_WINRATE_50: 53
ROLL_HOME_WINRATE_100: 52
ROLL_AWAY_WINRATE_100: 53
ROLL_HOME_WINRATE_200: 52
ROLL_AWAY_WINRATE_200: 53
HOME_WIN_STREAK: 0
AWAY_WIN_STREAK: 0
ROLL_HOME_PTS_FOR_5: 1701
ROLL_HOME_PTS_AGAINST_5: 1701
ROLL_AWAY_PTS_FOR_5: 1393
ROLL_AWAY_PTS_AGAINST_5: 1393
ROLL_HOME_PTS_FOR_10: 54
ROLL_HOME_PTS_AGAINST_10: 54
ROLL_AWAY_PTS_FOR_10: 53
ROLL_AWAY_PTS_AGAINST_10: 53
ROLL_HOME_PTS_FOR_25: 52
ROLL_HOME_PTS_AGAINST_25: 52
ROLL_AWAY_PTS_FOR_25: 53
ROLL_AWAY_PTS_AGAINST_25: 53
ROLL_HOME_PTS_FOR_50: 52
ROLL_HOME_PTS_AGAINST_50: 52
ROLL_AWAY_PTS_FOR_50: 53
ROLL_AWAY_PTS_AGAINST_50: 53
ROLL_HOME_PTS_FOR_100: 52
ROLL_HOME_PTS_AGAINST_100: 52
ROLL_AWAY_PTS_FOR_100: 53
ROLL_AWAY_PTS_AGAINST_100: 53
ROLL_HOME_PTS_FOR_200: 52
ROLL_HOME_PTS_AGAINST_

In [ ]:

#display full columns
pd.set_option('display.max_column', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_seq_items', None)
pd.set_option('display.max_colwidth', 500)
pd.set_option('expand_frame_repr', True)

# Display the final cleaned dataset
print("Final cleaned dataset:")
toto = final_dataset.columns
print(toto)

Final cleaned dataset:
Index(['TEAM_ID', 'GAME_DATE', 'OPP_TEAM_ID', 'IS_WIN', 'IS_HOME',
       'ROLL_HOME_WINRATE_5', 'ROLL_AWAY_WINRATE_5', 'ROLL_HOME_WINRATE_10',
       'ROLL_AWAY_WINRATE_10', 'ROLL_HOME_WINRATE_25', 'ROLL_AWAY_WINRATE_25',
       'ROLL_HOME_WINRATE_50', 'ROLL_AWAY_WINRATE_50', 'ROLL_HOME_WINRATE_100',
       'ROLL_AWAY_WINRATE_100', 'ROLL_HOME_WINRATE_200',
       'ROLL_AWAY_WINRATE_200', 'HOME_WIN_STREAK', 'AWAY_WIN_STREAK',
       'ROLL_HOME_PTS_FOR_5', 'ROLL_HOME_PTS_AGAINST_5', 'ROLL_AWAY_PTS_FOR_5',
       'ROLL_AWAY_PTS_AGAINST_5', 'ROLL_HOME_PTS_FOR_10',
       'ROLL_HOME_PTS_AGAINST_10', 'ROLL_AWAY_PTS_FOR_10',
       'ROLL_AWAY_PTS_AGAINST_10', 'ROLL_HOME_PTS_FOR_25',
       'ROLL_HOME_PTS_AGAINST_25', 'ROLL_AWAY_PTS_FOR_25',
       'ROLL_AWAY_PTS_AGAINST_25', 'ROLL_HOME_PTS_FOR_50',
       'ROLL_HOME_PTS_AGAINST_50', 'ROLL_AWAY_PTS_FOR_50',
       'ROLL_AWAY_PTS_AGAINST_50', 'ROLL_HOME_PTS_FOR_100',
       'ROLL_HOME_PTS_AGAINST_100', 'ROLL_AWAY_PTS_FOR

: 

In [47]:
#store end time of notebook
end_time = datetime.datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-05-25 11:52:18.819528
Total time:  0:01:53.671820
